# Session X: Lab — Building an Interactive Personal Budget Manager

**Goal:** Combine everything from Sessions I–IX into one complete application.

---

## Exercise 1: Define the Transaction Model

Create a `Transaction` class with:
- `INCOME_CATEGORIES` and `EXPENSE_CATEGORIES` class attributes
- `__init__`: amount, category, description, is_income (default False), auto-set date. Raise `ValueError` if the amount is not a positive, finite number (use `math.isfinite()`) or if the category is not in the list for that transaction type
- `to_dict()` and `@classmethod from_dict()` for serialization
- `__repr__` for readable output
- Type hints on all methods

### How to approach this code

A Transaction class is a blueprint for one record. The two category lists are shared class data, while `__init__` checks the amount and category before storing each object's values; `to_dict()` and `from_dict()` provide the two directions between objects and JSON-friendly dictionaries. `__repr__()` makes objects readable in debugging. The test checks that values survive a serialization round trip and that invalid data is rejected.

In [1]:
import math
from datetime import datetime


class Transaction:
    INCOME_CATEGORIES = ["Salary", "Freelance", "Gift", "Other income"]
    EXPENSE_CATEGORIES = ["Food", "Transport", "Entertainment", "Housing", "Utilities", "Other"]

    def __init__(self, amount: float, category: str, description: str,
                 is_income: bool = False) -> None:
        if (not isinstance(amount, (int, float)) or isinstance(amount, bool)
                or not math.isfinite(amount) or amount <= 0):
            raise ValueError(f"Amount must be a positive, finite number, got {amount!r}")

        allowed = self.INCOME_CATEGORIES if is_income else self.EXPENSE_CATEGORIES
        if category not in allowed:
            kind = "income" if is_income else "expense"
            raise ValueError(
                f"{category!r} is not a valid {kind} category. "
                f"Choose from: {', '.join(allowed)}"
            )

        self.amount = float(amount)
        self.category = category
        self.description = description
        self.is_income = is_income
        self.date = datetime.now().strftime("%Y-%m-%d")


    def to_dict(self) -> dict:
        return {
            "amount": self.amount, "category": self.category,
            "description": self.description, "is_income": self.is_income,
            "date": self.date,
        }

    @classmethod
    def from_dict(cls, data: dict) -> Transaction:
        t = cls(data["amount"], data["category"], data["description"], data["is_income"])
        t.date = data["date"]
        return t

    def __repr__(self) -> str:
        kind = "INCOME" if self.is_income else "EXPENSE"
        return f"[{kind}] {self.category}: ${self.amount:.2f} - {self.description}"


# Test
t1 = Transaction(25.50, "Food", "Lunch")
print(t1)
t2 = Transaction(3000, "Salary", "Monthly", is_income=True)
print(t2)
data = t1.to_dict()
t1_copy = Transaction.from_dict(data)
print(f"Round-trip: {t1.amount == t1_copy.amount}")
try:
    Transaction(-5, "Food", "Refund?")
except ValueError as error:
    print("Rejected:", error)
try:
    Transaction(10, "Salary", "Not an expense")
except ValueError as error:
    print("Rejected:", error)

[EXPENSE] Food: $25.50 - Lunch
[INCOME] Salary: $3000.00 - Monthly
Round-trip: True
Rejected: Amount must be a positive, finite number, got -5
Rejected: 'Salary' is not a valid expense category. Choose from: Food, Transport, Entertainment, Housing, Utilities, Other


---
## Exercise 2: Input Validation Functions

Write:
- `get_valid_amount(prompt)` — positive, finite number (reject `nan` and `inf` with `math.isfinite()`), catches ValueError
- `get_valid_category(categories)` — numbered menu, validates int input
- `get_valid_description()` — rejects empty strings

### How to approach this code

Validation first converts input text, then checks whether the converted value follows the app's rules. A retry loop lets users recover from nonnumeric amounts; separate checks reject zero, negative, non-finite (`nan`, `inf`) or out-of-range values. Reusable helpers keep those rules consistent across menu options.

In [3]:
import builtins as _b; _real_input = _b.input; _b.input = lambda p='': '10'
import math


def get_valid_amount(prompt: str) -> float:
    """Keep asking until the user enters a positive number."""
    while True:
        try:
            value = float(input(prompt))
            if not math.isfinite(value) or value <= 0:
                print("Error: Amount must be a positive number.")
                continue
            return value
        except ValueError:
            print("Error: Please enter a valid number.")


def get_valid_category(categories: list[str]) -> str:
    """Display a numbered menu and return the selected category."""
    while True:
        print("\nCategories:")
        for i, cat in enumerate(categories, 1):
            print(f"  [{i}] {cat}")
        try:
            choice = int(input("Select a category: "))
            if 1 <= choice <= len(categories):
                return categories[choice - 1]
            print(f"Error: Choose between 1 and {len(categories)}.")
        except ValueError:
            print("Error: Enter a number.")


def get_valid_description() -> str:
    """Keep asking until the description is not empty."""
    while True:
        text = input("Description: ").strip()
        if text:
            return text
        print("Error: Description cannot be empty.")



# Uncomment to test:
print(get_valid_amount("Amount: $"))
_b.input = _real_input

Error: Amount must be a positive number.
Error: Please enter a valid number.
10.0


---
## Exercise 3: JSON Persistence

Write `save_transactions(transactions, filepath)` and `load_transactions(filepath)`.
- Save through a temporary file and `os.replace()`, so a failed write never destroys the old file.
- A missing file (`FileNotFoundError`) is a normal first run: return `[]`.
- A file that is not valid JSON (`JSONDecodeError`), or valid JSON that is not a list, is damaged: move it to `filepath + ".bak"`, print a warning and return `[]`.

### How to approach this code

The save and load functions make in-memory transactions persist between program runs. Writing to a temporary file and swapping it in with `os.replace()` keeps the old file intact if the write fails. `FileNotFoundError` means a first run, while a damaged file is set aside as `.bak` so it is never overwritten. The test uses temporary paths to avoid changing real data.

In [3]:
import json
import os


def save_transactions(transactions: list[dict], filepath: str = "data/budget.json") -> None:
    """Write to a temporary file first, then swap it into place."""
    # TODO: write a temporary file, then os.replace() it into place
    os.makedirs(os.path.dirname(filepath) or ".", exist_ok=True)
    temp_path = filepath + ".tmp"
    with open(temp_path, "w", encoding="utf-8") as f:
        json.dump(transactions, f, indent=2)
    os.replace(temp_path, filepath)   # the old file stays intact until here
    print(f"Saved {len(transactions)} transactions to {filepath}.")


def load_transactions(filepath: str = "data/budget.json") -> list[dict]:
    """Load saved records; set a damaged file aside instead of losing it."""
    # TODO: missing file -> []; damaged or wrong shape -> move to .bak, warn, []
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)
    except FileNotFoundError:
        print("No saved data yet. Starting fresh.")
        return []
    except json.JSONDecodeError:
        data = None   # not valid JSON, handled below

    if not isinstance(data, list):   # bad JSON or the wrong shape
        backup = filepath + ".bak"
        os.replace(filepath, backup)
        print(f"Warning: {filepath} is damaged; moved it to {backup}.")
        return []
    return data


# Test
test_data = [{"amount": 50, "category": "Food", "description": "Groceries", "is_income": False, "date": "2026-10-01"}]
save_transactions(test_data, "/tmp/test_budget.json")
loaded = load_transactions("/tmp/test_budget.json")
print(loaded)
print(load_transactions("/tmp/nonexistent.json"))
with open("/tmp/damaged_budget.json", "w", encoding="utf-8") as f:
    f.write('{"amount": 5,')
print(load_transactions("/tmp/damaged_budget.json"))
print(os.path.exists("/tmp/damaged_budget.json.bak"))

Saved 1 transactions to /tmp/test_budget.json.
[{'amount': 50, 'category': 'Food', 'description': 'Groceries', 'is_income': False, 'date': '2026-10-01'}]
No saved data yet. Starting fresh.
[]
[]
True


---
## Exercise 4: CSV Export

Write `export_to_csv(transactions, filepath)` using csv.DictWriter. Create the folder if it is missing, open the file with `newline=""` and `encoding="utf-8"`, and handle an empty list.

### How to approach this code

CSV export maps each transaction dictionary to a row with a stable set of columns. `writeheader()` labels the fields and `writerows()` writes records; the empty-list check prevents a false success message. Reading the temporary export back verifies the actual output.

In [4]:
import csv
import os


def export_to_csv(transactions: list[dict], filepath: str = "data/export.csv") -> None:
    """Write the transactions to a CSV file with a fixed column order."""
    # TODO
    if not transactions:
        print("Nothing to export.")
        return

    fieldnames = ["date", "category", "description", "amount", "is_income"]
    os.makedirs(os.path.dirname(filepath) or ".", exist_ok=True)   # create the folder if missing
    with open(filepath, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(transactions)
    print(f"Exported {len(transactions)} transactions to {filepath}")


# Test
test_data = [{"date":"2026-10-01","category":"Food","description":"Groceries","amount":50.0,"is_income":False}]
export_to_csv(test_data, "/tmp/test_export.csv")
with open("/tmp/test_export.csv", encoding="utf-8") as f:
    print(f.read())
export_to_csv([])

Exported 1 transactions to /tmp/test_export.csv
date,category,description,amount,is_income
2026-10-01,Food,Groceries,50.0,False

Nothing to export.


---
## Exercise 5: Summary Report

Write `generate_report(transactions)` that calculates income/expenses/balance, groups expenses by category and shows percentages. Handle an empty list, and print no "Expenses by Category" heading for an income-only list.

### How to approach this code

The report calculates income, expenses, balance, and totals by category from the same transaction list. Generator expressions select records before summing; a dictionary accumulates category totals. Print the category block only when there is at least one expense: then `expenses > 0`, so the percentage division is safe, and an income-only report has no empty heading.

In [5]:
def generate_report(transactions: list[dict]) -> None:
    """Print income, expenses, balance and the expenses grouped by category."""
    # TODO
    if not transactions:
        print("No transactions to report.")
        return

    income = sum(t["amount"] for t in transactions if t["is_income"])
    expenses = sum(t["amount"] for t in transactions if not t["is_income"])
    balance = income - expenses

    by_category = {}
    for transaction in transactions:
        if not transaction["is_income"]:
            category = transaction["category"]
            by_category[category] = by_category.get(category, 0) + transaction["amount"]

    print("\n" + "=" * 50)
    print("         BUDGET SUMMARY REPORT")
    print("=" * 50)
    print(f"  Total Income:   ${income:>10.2f}")
    print(f"  Total Expenses: ${expenses:>10.2f}")
    print(f"  Balance:        ${balance:>10.2f}")
    if by_category:   # no heading for an income-only list
        print("-" * 50)
        print("  Expenses by Category:")
        for category, total in sorted(by_category.items(), key=lambda item: item[1], reverse=True):
            percentage = total / expenses * 100   # by_category is not empty, so expenses > 0
            print(f"    {category:<20} ${total:>8.2f}  ({percentage:.1f}%)")
    print("=" * 50)


test_data = [
    {"amount": 3000, "category": "Salary", "description": "Monthly", "is_income": True},
    {"amount": 800, "category": "Housing", "description": "Rent", "is_income": False},
    {"amount": 150, "category": "Food", "description": "Groceries", "is_income": False},
    {"amount": 75, "category": "Food", "description": "Restaurants", "is_income": False},
    {"amount": 50, "category": "Transport", "description": "Metro", "is_income": False},
]
generate_report(test_data)
generate_report([])
generate_report(test_data[:1])   # income only: no "Expenses by Category" heading


         BUDGET SUMMARY REPORT
  Total Income:   $   3000.00
  Total Expenses: $   1075.00
  Balance:        $   1925.00
--------------------------------------------------
  Expenses by Category:
    Housing              $  800.00  (74.4%)
    Food                 $  225.00  (20.9%)
    Transport            $   50.00  (4.7%)
No transactions to report.

         BUDGET SUMMARY REPORT
  Total Income:   $   3000.00
  Total Expenses: $      0.00
  Balance:        $   3000.00


---
## Exercise 6: Main Application Loop

Wire everything together with a menu: Add Income, Add Expense, View All, Report, Export CSV, Save & Quit, Quit.
- At start-up, load the saved records and convert them with `clean_records()`, which skips (and reports) any record that fails validation.
- `ask_transaction(is_income)` asks for the amount, a category from the right list, and a description.
- Report, export and save work with dictionaries: convert with `[t.to_dict() for t in transactions]`.
- Q asks before discarding unsaved changes; any other key prints an error message.

### How to approach this code

The main loop coordinates the application: it displays choices and calls the relevant validation, model, reporting, export, and storage operations. Keep those implementations in their own functions so each menu branch remains easy to read. Saving before exit differs from quitting without saving, so Q should warn when there are unsaved changes.

In [6]:
def clean_records(records: list[dict]) -> list[Transaction]:
    """Turn saved records into Transactions, skipping invalid ones."""
    # TODO: from_dict() each record; report and skip the ones that raise an error
    transactions = []
    for number, record in enumerate(records, 1):
        try:
            transaction = Transaction.from_dict(record)
        except (KeyError, TypeError, ValueError) as error:
            print(f"Skipping record {number} ({type(error).__name__}: {error})")
            continue
        transactions.append(transaction)
    return transactions


def ask_transaction(is_income: bool) -> Transaction:
    """Ask for one income or expense, using the validation helpers."""
    # TODO: amount, a category from the right list, description
    if is_income:
        categories = Transaction.INCOME_CATEGORIES
    else:
        categories = Transaction.EXPENSE_CATEGORIES
    amount = get_valid_amount("Amount: $")
    category = get_valid_category(categories)
    description = get_valid_description()
    return Transaction(amount, category, description, is_income)


def main() -> None:
    transactions = clean_records(load_transactions())  # TODO: load the saved records and clean them
    unsaved = False
    while True:
        print("\n=== PERSONAL BUDGET MANAGER ===")
        print("[1] Add Income  [2] Add Expense  [3] View All")
        print("[4] Report  [5] Export CSV  [6] Save & Quit  [Q] Quit")
        choice = input("Select: ").strip().lower()
        if choice == "1":
            transaction = ask_transaction(is_income=True)
            transactions.append(transaction)
            unsaved = True
            print(f"Added: {transaction}")
        elif choice == "2":
            transaction = ask_transaction(is_income=False)
            transactions.append(transaction)
            unsaved = True
            print(f"Added: {transaction}")
        elif choice == "3":
            if not transactions:
                print("No transactions yet.")
            for transaction in transactions:
                print(transaction)
        elif choice == "4":
            generate_report([t.to_dict() for t in transactions])
        elif choice == "5":
            export_to_csv([t.to_dict() for t in transactions])
        elif choice == "6":
            # TODO: save and break
            save_transactions([t.to_dict() for t in transactions])
            break
        elif choice == "q":
            # TODO: if there are unsaved changes, ask before quitting
            if unsaved:
                answer = input("Quit without saving? (y/n): ").strip().lower()
                if answer != "y":
                    continue
            print("Goodbye!")
            break
        else:
            print("Invalid option: choose 1-6 or Q.")


# Test clean_records: only the first record is valid
records = [
    {"amount": 12.5, "category": "Food", "description": "Lunch", "is_income": False, "date": "2026-10-01"},
    {"amount": -3, "category": "Food", "description": "Typo", "is_income": False, "date": "2026-10-02"},
    {"amount": 40, "category": "Pets", "description": "Vet", "is_income": False, "date": "2026-10-03"},
    {"amount": 8, "category": "Food", "description": "No date", "is_income": False},
]
print(clean_records(records))

# Uncomment to run (Ctrl-D or the stop button ends the input):
# main()

Skipping record 2 (ValueError: Amount must be a positive, finite number, got -3)
Skipping record 3 (ValueError: 'Pets' is not a valid expense category. Choose from: Food, Transport, Entertainment, Housing, Utilities, Other)
Skipping record 4 (KeyError: 'date')
[[EXPENSE] Food: $12.50 - Lunch]
